# LangGraph AI Research Pipeline Demo

This notebook demonstrates the new LangGraph-based architecture with PageIndex caching.

In [ ]:
import sys
sys.path.insert(0, r'C:\\Shrey_Projs\\AI_ML_Projs\\test_openrouter_nvidia')

from modules import LangGraphApp, PageIndex

## Initialize the App

In [ ]:
# Create app with default PageIndex cache
app = LangGraphApp()

# Check cache stats
print("Cache Stats:", app.get_cache_stats())

## Example 1: Simple Query (routes to worker model)

In [ ]:
result = app.invoke("What is the capital of France?")

print(f"Content: {result['content']}")
print(f"Model: {result['model']}")
print(f"Tokens: {result['tokens']}")
print(f"Time: {result['time']}s")
print(f"Task Type: {result['classification']['task_type']}")
print(f"Quality: {result['validation']['quality_score']}/10")

## Example 2: Complex Query (routes to orchestrator)

In [ ]:
result = app.invoke("Explain quantum entanglement and its implications for cryptography")

print(f"Content: {result['content'][:300]}...")
print(f"Model: {result['model']}")
print(f"Time: {result['time']}s")
print(f"Task Type: {result['classification']['task_type']}")

## Example 3: Research Query (search + cache + LLM)

In [ ]:
result = app.invoke("What are the latest developments in AI in 2026?")

print(f"Content: {result['content'][:500]}...")
print(f"\nSources: {result['sources']}")
print(f"\nCache Stats: {result['cache_stats']}")
print(f"Validation: {result['validation']['quality_score']}/10")

## Example 4: Cache Demonstration (same query = cache hit)

In [ ]:
# Run same query again - should use cached content
result2 = app.invoke("What are the latest developments in AI in 2026?")

print(f"Cache Stats: {result2['cache_stats']}")
print(f"Time: {result2['time']}s (should be faster)")

## Example 5: Conversation Memory

In [ ]:
# Use thread_id to maintain conversation context
app_with_memory = LangGraphApp(memory=True)

result1 = app_with_memory.invoke("What is Python?", thread_id="conv_1")
result2 = app_with_memory.invoke("What are its main use cases?", thread_id="conv_1")

print("Query 1:", result1['content'][:200])
print("\nQuery 2 (with context):", result2['content'][:200])

## Cache Management

In [ ]:
# View cache stats
stats = app.get_cache_stats()
print(f"Total entries: {stats['total_entries']}")
print(f"DB size: {stats['db_size_mb']} MB")
print(f"Default TTL: {stats['default_ttl_seconds']}s")

# Clear cache if needed
# cleared = app.clear_cache()
# print(f"Cleared {cleared} entries")

## Architecture Overview

```
User Query
    |
    v
[Classify Node] (Orchestrator)
    |
    |-- simple --> [Worker LLM] --> [Validate]
    |-- complex --> [Orchestrator LLM] --> [Validate]
    |-- research --> [Search] --> [Extract*] --> [LLM Synthesize] --> [Validate]
                                                       |
                                                [*Cache Check]
                                                       |
                                            hit --> use cached
                                            miss --> Jina AI --> store in cache
    |
    v
[Result]
```